# B4 · Tinh chỉnh (finetune) mạng ẢNH + CHỮ trên GPU  (20–60 phút)

Học tiếp cả bộ đọc ảnh và bộ đọc chữ trên dữ liệu đề, thường **tăng điểm rõ** so với B3. GPU < 24 GB.
In ra accuracy trên 15% dữ liệu train giữ lại sau mỗi vòng. So với điểm B3 trước khi nộp.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())          # thư mục notebooks (chứa b2lib.py)
import numpy as np, pandas as pd
import b2lib as L
print("thư mục làm việc:", L.WORK)

In [ ]:
df = L.doc_bang_chuan()
itr, ite = (df.split == "train").to_numpy(), (df.split == "test").to_numpy()
y = df.loc[itr, "label"].astype(str).to_numpy()
ids_tr, ids_te = df.loc[itr, "id"].to_numpy(), df.loc[ite, "id"].to_numpy()
print(itr.sum(), "train |", ite.sum(), "test |", len(set(y)), "lớp")

## ✏️ Chọn cấu hình

| ANH | CHU | VRAM ước lượng (batch 32) | ghi chú |
|---|---|---|---|
| `"clip"` | `"e5"` | ~6 GB | thử đầu tiên, nhanh |
| `"siglip"` | `"e5_b"` | ~12 GB | mạnh hơn |
| `"clip_l"` | `"e5_b"` | ~20 GB | lớn nhất; hết VRAM thì BATCH = 16 |
| `None` | `"bkai"` / `"e5_b"` | ~6 GB | bài CHỈ có chữ |
| `"dinov2_b"` | `None` | ~8 GB | bài CHỈ có ảnh |

Báo `CUDA out of memory` → giảm BATCH (32 → 16 → 8).

In [ ]:
ANH, CHU = "clip", "e5"
EPOCHS, BATCH, PHUT = 5, 32, 40
giu = np.load(L.WORK / "giu_lai.npy") if (L.WORK / "giu_lai.npy").exists() else np.ones(itr.sum(), bool)
train_df = df[itr].reset_index(drop=True)[giu].reset_index(drop=True)
test_df = df[ite].reset_index(drop=True)
acc, tp, cl, path = L.tinh_chinh(train_df, test_df, anh=ANH, chu=CHU, epochs=EPOCHS, batch=BATCH, phut=PHUT)

In [ ]:
FILE_MAU = ""                          # ✏️
COT_ID, COT_NHAN = "id", "label"       # ✏️
ten = f"tinh_chinh_{ANH}_{CHU}"
L.luu_xac_suat(ten, ids_te, tp, cl)
L.ghi_nop(ids_te, cl[tp.argmax(1)], pd.DataFrame(tp, columns=cl), FILE_MAU or None, COT_ID, COT_NHAN, ten="nop_" + ten)